In [6]:
#######simulate DOSY data########
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from io import StringIO
from scipy.optimize import curve_fit

In [7]:
def import_data(input_file): #imports data to dataframe, filters out all lines starting with # or empty lines
#removes the last column of dataframe as it appears to be empty
   
    valid_lines = []  # Create list to store valid lines

    with open(input_file, 'r') as infile: # rad the file
        for line in infile:
            if line.strip() and not line.startswith('#'): # Check if the line is not empty and does not start with '#'
                valid_lines.append(line.strip())

    # to use read.csv() (and all the benefits) a file like object has to be created. This is done with StringIO
    valid_data = "\n".join(valid_lines) #Join the valid lines into a single string with newline characters
    data_io = StringIO(valid_data)  # Use StringIO to create a file-like object from the string

    df = pd.read_csv(data_io, header=None, index_col=0, sep=";")  # Read the data into a DataFrame
    # maybe has to be customized
    dataframe = df.iloc[:, :-1] #removes last column, as this one is empty
    dataframe = dataframe.iloc[0:] #removes the first row as this one is empty
    #dataframe = dataframe.replace("", np.nan).dropna(how='all') #removes completely empty rows

    return dataframe

def cov2corr(covalence_matrix):
    std_devs = np.sqrt(np.diag(covalence_matrix))
    correlation_matrix = covalence_matrix / np.outer(std_devs, std_devs)
    return std_devs

In [8]:
####fitting functions###

def sigmoidal_func_var_p30(Gradient_length, D):
    return np.exp(-((2.678*10**8)**2 * gradient_strength**2 * Gradient_length**2 * (2/np.pi)**2 * D * (diff_time - Gradient_length / 3))) 

def sigmoidal_func_DOSY(gradient_strength, D):
    return np.exp(-((2.678*10**8)**2 * gradient_strength**2 * Gradient_length**2 * (2/np.pi)**2 * D * (diff_time - Gradient_length / 3))) 


def sigmoidal_func_comp(sep_power, D):
    return np.exp(-((2.678*10**8)**2 * sep_power**2 * (2/np.pi)**2 * D * (diff_time - Gradient_length / 3))) 


# def sigmoidal_func_var_p30_2(Gradient_length, gradient_strength, D):
#     return np.exp(-((2.678*10**8)**2 * gradient_strength**2 * Gradient_length**2 * (2/np.pi)**2 * D * (diff_time - Gradient_length / 3))) 

# def sigmoidal_func_DOSY_2(gradient_strength, Gradient_length, D):
#     return np.exp(-((2.678*10**8)**2 * gradient_strength**2 * Gradient_length**2 * (2/np.pi)**2 * D * (diff_time - Gradient_length / 3))) 

In [9]:

p30_time_comp = np.array([162.7906977, 581.39534892, 1000.00000014, 1418.60465136, 1837.20930258, 2255.8139538,  2674.41860502, 3093.02325625]) #gradient length of p30 in us
delta_comp = p30_time_comp * 2 / 1000000 #delta in s, can be used for fitting
G_comp = np.array([0.02966084, 0.10593156, 0.18220228, 0.258473, 0.33474372, 0.41101444, 0.48728516, 0.56355588]) # Gradient strength in T/m, 5-95 % 8 increments

In [10]:
####actual processing###

y = (2.675 * 10**8) #gyromagnetic ratio proton (s-1T-1)
diff_time = 0.05 #diffusion time in seconds

gradient_strength = 0.563555883 / 0.95 * 0.3071 #Gradient strength 100% * 30.71 %
Gradient_length = 0.002 #2*p30 in seconds Zita

initial_guess = [ 10**-9]

#read in data
df_var_p30 = import_data(r"C:\Users\bruno\Documents\Studium\MP_Kovermann\Processing\Ethylbenzol\Integrals\Zita\EtBn-var-p30-Zita")
df_DOSY = import_data(r"C:\Users\bruno\Documents\Studium\MP_Kovermann\Processing\Ethylbenzol\Integrals\Zita\EtBn-var-gpz6-Zita")

#var_p30_1D_data = np.array([746009243.3,599332990.6,377984388.9,187755990.9,74127830.47,23920785.95,6242500.18,1434092.52])#,378778923.1])
#var_p30_1D_data = var_p30_1D_data / max(var_p30_1D_data)


#DOSY:   EB_DOSY_sep_power_580us_305
#        EB_DOSY_sep_power_1410us_303

#var_p30:    EB_var_p30_sep_power_17_304
#            EB_var_p30_sep_power_43_302

#selecting peak and normalizing
var_p30_data = df_var_p30.iloc[:,3] / max(df_var_p30.iloc[:,3])
DOSY_data = df_DOSY.iloc[:,3] / max(df_DOSY.iloc[:,3])

#print(var_p30_data)

#fitting
popt_var_p30, pcov_var_p30 = curve_fit(sigmoidal_func_var_p30, delta_comp, var_p30_data, p0 = initial_guess) #var_p30
popt_DOSY, pcov_DOSY = curve_fit(sigmoidal_func_DOSY, G_comp, DOSY_data, p0 = initial_guess) #DOSY

#popt_var_p30_1D, pcov_var_p30_1D = curve_fit(sigmoidal_func_var_p30, delta_comp, var_p30_1D_data, p0 = initial_guess) #var_p30

#uncertainties of fit
var_p30_uncert = cov2corr(pcov_var_p30)
DOSY_uncert = cov2corr(pcov_DOSY)

#var_p30_uncert_1D = cov2corr(pcov_var_p30_1D)

#print results of the fit
print("D (vartiable Gradient length):",  popt_var_p30 , "±", var_p30_uncert)
print("D (vartiable Gradient strength):",  popt_DOSY , "±", DOSY_uncert)

#print("D (vartiable Gradient length 1D):",  popt_var_p30_1D , "±", var_p30_uncert_1D)


# #plot of fitting results
# plt.scatter(delta_comp, var_p30_data, color="red", label="pseudo_2D_data")

var_p30_x_fit = np.linspace(min(delta_comp), max(delta_comp), 100) #array (for x-axis) to make fit look smoother. not smooth enough --> more points
var_p30_y_fit = sigmoidal_func_var_p30(var_p30_x_fit, *popt_var_p30) #array (for y-axis) to plot the fit

# plt.plot(var_p30_x_fit, var_p30_y_fit, color="magenta")
# plt.xlabel('Gradient length / s')
# plt.ylabel('Integral / a.u.')


# ##1Ds##
# var_p30_1D_y_fit = sigmoidal_func_var_p30(var_p30_x_fit, *popt_var_p30_1D) #array (for y-axis) to plot the fit

# plt.scatter(delta_comp, var_p30_1D_data, color="blue", label="1D_data")
# plt.plot(var_p30_x_fit, var_p30_1D_y_fit, color="lightskyblue")
# plt.legend()


# plt.figure()  # Creates a new figure

# plt.scatter(G_comp, DOSY_data, color="black")
DOSY_x_fit = np.linspace(min(G_comp), max(G_comp), 100) #array (for x-axis) to make fit look smoother. not smooth enough --> more points
DOSY_y_fit = sigmoidal_func_DOSY(DOSY_x_fit, *popt_DOSY) #array (for y-axis) to plot the fit

# plt.plot(DOSY_x_fit, DOSY_y_fit, color="red")
# plt.xlabel('Gradient strength / T/m')
# plt.ylabel('Integral / a.u.')


#plt.savefig('3D_fit_data_test/Ethylbenzol/results/approach_comparison_DOSY.png', dpi=300)

plt.figure()  # Creates a new figure

sep_power = G_comp * Gradient_length
x_fit_comp = np.linspace(min(sep_power), max(sep_power), 100)

plt.scatter(sep_power, DOSY_data, color="black")
plt.scatter(sep_power, var_p30_data, color="black")
#plt.scatter(sep_power, var_p30_1D_data, color="black")
plt.xlabel(r'$\mathit{sep\_power}$ (G$^2 \cdot \Delta^2$) / $\frac{T^2 s^2}{m^2}$')
plt.ylabel('Integral / a.u.')


plt.plot(x_fit_comp, DOSY_y_fit, color="red", label = "DOSY_AA")
plt.plot(x_fit_comp, var_p30_y_fit, color="black", label = "var_p30")
#plt.plot(x_fit_comp, var_p30_1D_y_fit, color="lightskyblue", label="var_p30_1D_data")

plt.legend()



#plt.savefig('3D_fit_data_test/Ethylbenzol/results/approach_comparison_3_datasets.png', dpi=300)  # Saves the plot as a PNG file with 300 dpi
#plt.close()  # Close the plot to free u


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\bruno\\Documents\\Studium\\MP_Kovermann\\Processing\\Ethylbenzol\\Integrals\\Zita\\EtBn-var-p30-Zita'